# Maia Lite Chat 0.3.0 — Maia Corpus Builder v8
## Scalable, provenance-first multilingual data pipeline

**Target:** 354,823,168 parameters; **21,289,390,080 processed tokens** (~3× the 20 tokens/parameter Chinchilla heuristic). This is an **exposure budget**, **not a claim of unique tokens**. Language exposure targets: PT 40%, EN 35%, ES 25%. Domain exposure hypotheses: general language 30%, dialogue 20%, logic 25%, educational 15%, creative 10%.

This Colab notebook builds a **small, safe pilot by default**. It can ingest local v4/v7 files, externally provided JSONL files, and optionally approved Hugging Face streaming datasets. It keeps candidate/eligible/training artifacts separate, persists exact-dedup state in SQLite, creates source-grouped splits, writes compressed shards, records licenses and SHA-256, and computes a training-exposure plan. No dataset is automatically approved for training. No massive downloads start without an explicit opt-in.

**Important limitations:** automated filters do not establish copyright compliance, linguistic correctness, near-duplicate freedom or absence of benchmark contamination. A final tokenizer requires approved, representative data. Large-scale processing should eventually run on persistent storage/HPC rather than depending on a Colab session.

In [ ]:
# 1 — Install dependencies (run in Colab)
%pip -q install "datasets>=3.0,<5" "transformers>=4.44,<6" "tokenizers>=0.19" pandas tqdm requests

In [ ]:
# 2 — Configuration and Google Drive
from pathlib import Path
import os, json, re, csv, gzip, hashlib, sqlite3, random, math, time, unicodedata
from collections import defaultdict, Counter
from datetime import datetime, timezone
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab; edit ROOT for your environment.')

ROOT = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat')
V8 = ROOT / 'corpus_v8'
for sub in ('config','reports','shards','imports','cache'):
    (V8/sub).mkdir(parents=True, exist_ok=True)
SEED = 20261018
PILOT_MAX_DOCUMENTS_PER_SOURCE = 1000  # bounded, deterministic pilot
PILOT_MAX_TOKENS_TOTAL = 250_000        # only for eligible training export
SHARD_MAX_RECORDS = 2000
ENABLE_HF_STREAMING = False  # explicit opt-in after editing registry
ENABLE_LOCAL_IMPORTS = True
ENABLE_V4_LEGACY = True
ENABLE_TOKENIZER_EXPERIMENT = False
TARGET_TOKENS = 21_289_390_080
TARGET_LANG = {'pt': .40, 'en': .35, 'es': .25}
TARGET_DOMAIN = {'language': .30, 'dialogue': .20, 'logic': .25, 'educational': .15, 'creative': .10}
print('Output:', V8)

## A. Source registry and rights gate

The registry is **user-editable**. Every source has a `source_id`, `family_id`, `language`, `domain`, `format`, `path` or Hugging Face dataset metadata, a declared license, source URL, and human-reviewed eligibility fields. Existing v7 evidence is imported **as evidence only**, never interpreted as approval. `family_id` is a conservative grouping key for splitting (e.g., a Wikipedia language snapshot). Source-level approval is necessary but may be insufficient for mixed-license collections: per-document exceptions must be screened before enabling export.

To add contemporary dialogue/educational collections, place one or more `.jsonl` or `.txt` files under `corpus_v8/imports/<language>/<domain>/<collection>/`. A JSONL record should contain `text`, preferably `id`, `url`, `revision_id`, `license`, and `family_id`. Unknown metadata stays unknown; never fabricate it.

In [ ]:
# 3 — Bootstrap registry from local files; preserve prior edits
REGISTRY_FILE = V8/'config/source_registry_v8.json'
old = json.loads(REGISTRY_FILE.read_text(encoding='utf-8')) if REGISTRY_FILE.exists() else {}
registry = dict(old)

# Existing v4 texts and optional v5 educational sources
for base in (ROOT/'corpus_v4/inbox', ROOT/'corpus_v5/inbox'):
    if not base.exists(): continue
    for path in sorted(base.rglob('*')):
        if path.suffix.lower() not in ('.txt','.jsonl') or not path.is_file(): continue
        rel = path.relative_to(base)
        if len(rel.parts)<3 or rel.parts[0] not in TARGET_LANG or rel.parts[1] not in TARGET_DOMAIN: continue
        source_id = f'legacy/{rel.as_posix()}'
        if source_id in registry: continue
        # Prefer v4 over duplicated v5 paths
        if base.name == 'inbox' and 'corpus_v5' in str(base) and source_id in registry: continue
        registry[source_id] = dict(source_id=source_id, family_id=f'{rel.parts[0]}/{rel.parts[2]}',
            language=rel.parts[0], domain=rel.parts[1], format=path.suffix[1:],
            path=str(path), kind='local', source_url='', declared_license='',
            status='pending_review', review_complete=False, approved_for_training=False,
            reviewer='', review_evidence='', attribution='', rights_notes='',
            per_document_rights_checked=False)

if ENABLE_LOCAL_IMPORTS:
    for path in sorted((V8/'imports').rglob('*')):
        if not path.is_file() or path.suffix.lower() not in ('.txt','.jsonl'): continue
        rel=path.relative_to(V8/'imports')
        if len(rel.parts)<4 or rel.parts[0] not in TARGET_LANG or rel.parts[1] not in TARGET_DOMAIN: continue
        sid='import/'+rel.as_posix()
        registry.setdefault(sid, dict(source_id=sid, family_id='/'.join(rel.parts[:3]),
            language=rel.parts[0], domain=rel.parts[1], format=path.suffix[1:], path=str(path),
            kind='local', source_url='', declared_license='', status='pending_review',
            review_complete=False, approved_for_training=False, reviewer='', review_evidence='',
            attribution='', rights_notes='', per_document_rights_checked=False))
REGISTRY_FILE.write_text(json.dumps(registry,ensure_ascii=False,indent=2),encoding='utf-8')
print('Registered sources:',len(registry),'| Registry:',REGISTRY_FILE)

In [ ]:
# 4 — Optional Hugging Face dataset registry template (no automatic discovery/download)
HF_CONFIG_FILE = V8/'config/hf_sources_v8.json'
if not HF_CONFIG_FILE.exists():
    HF_CONFIG_FILE.write_text(json.dumps([
        {'source_id':'REPLACE_WITH_REVIEWED_DATASET', 'dataset_id':'ORG/DATASET',
         'subset':None, 'split':'train', 'text_field':'text', 'language':'pt',
         'domain':'dialogue', 'family_id':'reviewed-dialogue-family',
         'declared_license':'', 'source_url':'', 'revision':None,
         'status':'pending_review', 'review_complete':False,
         'approved_for_training':False, 'reviewer':'', 'review_evidence':'',
         'attribution':'', 'rights_notes':'', 'per_document_rights_checked':False}
    ],ensure_ascii=False,indent=2),encoding='utf-8')
print('HF config:',HF_CONFIG_FILE)
if ENABLE_HF_STREAMING:
    hf_items=json.loads(HF_CONFIG_FILE.read_text(encoding='utf-8'))
    for item in hf_items:
        if item['source_id'].startswith('REPLACE_') or item['dataset_id']=='ORG/DATASET':
            continue
        registry.setdefault(item['source_id'], {**item, 'kind':'hf', 'format':'hf'})
    REGISTRY_FILE.write_text(json.dumps(registry,ensure_ascii=False,indent=2),encoding='utf-8')
print('HF streaming enabled:',ENABLE_HF_STREAMING)

In [ ]:
# 5 — Import v7 evidence for reviewer convenience (NO automatic approval)
V7_DOC = ROOT/'corpus_v7/reports/document_inventory_v7.csv'
V7_SOURCE = ROOT/'corpus_v7/reports/source_review_queue_v7.csv'
for f in (V7_DOC,V7_SOURCE):
    if f.exists():
        frame=pd.read_csv(f,nrows=3)
        print(f.name, 'columns:', list(frame.columns))
    else: print('Not found:',f)
print('Use these records to verify URL/revision/rights before editing source_registry_v8.json.')

In [ ]:
# 6 — Validate rights gate. Explicit manual review is required.
registry=json.loads(REGISTRY_FILE.read_text(encoding='utf-8'))
REQUIRED=('reviewer','review_evidence','attribution','rights_notes','declared_license','source_url')
def eligible(src):
    return (src.get('status')=='approved' and src.get('review_complete') is True
            and src.get('approved_for_training') is True
            and src.get('per_document_rights_checked') is True
            and all(str(src.get(k,'')).strip() for k in REQUIRED))
approved_ids={sid for sid,s in registry.items() if eligible(s)}
flagged=[sid for sid,s in registry.items() if s.get('approved_for_training') and sid not in approved_ids]
print('Sources:',len(registry),'| Eligible:',len(approved_ids),'| Incomplete approvals:',flagged[:10])

## B. Streaming readers, deterministic document IDs, quality filters

Readers process one document at a time. `jsonl` accepts `text`, `content`, `extract`, or `body` and preserves known provenance fields. For very large books, a text file is split into bounded blocks, retaining the same `family_id`. For wiki JSONL with missing title/revision, identity is marked **unverified** rather than inferred from today's page. The local pilot caps documents per source.

In [ ]:
# 7 — Document readers
TEXT_FIELDS=('text','content','extract','body')
def utc_now(): return datetime.now(timezone.utc).isoformat()
def sha256(s): return hashlib.sha256(s.encode('utf-8')).hexdigest()
def normalize_text(text):
    text=unicodedata.normalize('NFC',str(text)).replace('\r\n','\n').replace('\r','\n')
    text=re.sub(r'[ \t]+',' ',text)
    return re.sub(r'\n{4,}','\n\n',text).strip()
def chunk_text(text,limit=5000):
    # Bounded blocks for large TXT files; no overlap
    for start in range(0,len(text),limit):
        block=text[start:start+limit].strip()
        if block: yield block

def local_documents(src):
    p=Path(src['path'])
    if not p.is_file(): return
    if p.suffix.lower()=='.txt':
        raw=p.read_text(encoding='utf-8',errors='replace')
        a=re.search(r'\*\*\*\s*START OF (?:THE|THIS) PROJECT GUTENBERG',raw,re.I)
        b=re.search(r'\*\*\*\s*END OF (?:THE|THIS) PROJECT GUTENBERG',raw,re.I)
        if a and b and a.end()<b.start(): raw=raw[a.end():b.start()]
        for i,part in enumerate(chunk_text(raw)):
            yield {'text':part,'id':f'{p.name}:part{i}','document_group':str(p)}
    else:
        with p.open(encoding='utf-8',errors='replace') as f:
            for i,line in enumerate(f):
                try: obj=json.loads(line)
                except json.JSONDecodeError: continue
                if isinstance(obj,str): obj={'text':obj}
                if not isinstance(obj,dict): continue
                txt=next((obj[k] for k in TEXT_FIELDS if isinstance(obj.get(k),str)),None)
                if txt:
                    yield {'text':txt,'id':str(obj.get('id') or obj.get('pageid') or f'line{i}'),
                        'title':obj.get('title'),'url':obj.get('url') or obj.get('source_url'),
                        'revision_id':obj.get('revision_id') or obj.get('revid'),
                        'license':obj.get('license'),'document_group':str(obj.get('pageid') or obj.get('title') or f'line{i}')}

def hf_documents(src):
    from datasets import load_dataset
    kwargs={'path':src['dataset_id'],'split':src.get('split','train'),'streaming':True}
    if src.get('subset'):kwargs['name']=src['subset']
    if src.get('revision'):kwargs['revision']=src['revision']
    ds=load_dataset(**kwargs)
    for i,obj in enumerate(ds):
        txt=obj.get(src.get('text_field','text'))
        if isinstance(txt,str) and txt.strip():
            yield {'text':txt,'id':str(obj.get('id',i)), 'title':obj.get('title'),
                'url':obj.get('url'), 'revision_id':obj.get('revision_id'),
                'license':obj.get('license'), 'document_group':str(obj.get('document_id') or obj.get('id') or i)}

def documents(src):
    return hf_documents(src) if src.get('kind')=='hf' else local_documents(src)

In [ ]:
# 8 — Quality checks (heuristics; flagged text remains candidate-only if ineligible)
def quality_flags(text):
    flags=[]
    if len(text)<120: flags.append('too_short')
    if len(text)>100000: flags.append('too_long')
    if len(text)>0 and sum(c.isalpha() for c in text)/len(text)<.40: flags.append('low_alpha')
    if len(text)>0 and text.count('�')/len(text)>.002: flags.append('replacement_chars')
    words=re.findall(r'\w+',text.casefold())
    if len(words)>40 and len(set(words))/len(words)<.08: flags.append('low_lexical_diversity')
    return flags

# Token counts are estimates until the final Maia tokenizer is selected.
from transformers import AutoTokenizer
proxy=AutoTokenizer.from_pretrained('gpt2')
def proxy_tokens(text): return len(proxy.encode(text,add_special_tokens=False))

## C. Persistent exact deduplication, audit and pilot export

SQLite stores seen hashes across Colab restarts. The pipeline is **restart-safe at source granularity**: it commits a source's hash insertions, inventory rows and eligible shards only after that source is fully processed. A changed registry/source invalidates previous runs: use a fresh run ID. A pilot is deliberately small; to scale up, run bounded batches with persistent storage and stable source revisions.

In [ ]:
# 9 — Create a fresh run folder (run ID explicitly pinned)
RUN_ID='pilot_v8_001'  # CHANGE to a NEW value for a fresh run or changed input
RUN=V8/'runs'/RUN_ID
RUN.mkdir(parents=True,exist_ok=True)
DB=RUN/'seen.sqlite'
conn=sqlite3.connect(DB)
conn.execute('CREATE TABLE IF NOT EXISTS seen (hash TEXT PRIMARY KEY, source_id TEXT NOT NULL)')
conn.execute('CREATE TABLE IF NOT EXISTS completed (source_id TEXT PRIMARY KEY, fingerprint TEXT NOT NULL, stats_json TEXT NOT NULL)')
conn.commit()
print('Run:',RUN)

In [ ]:
# 10 — Source-level streaming audit and eligible-only pilot shards
# Invariant: never export candidate-only or unapproved records.
# Use a new RUN_ID if any source, approval decision, or parser logic changes.
from itertools import islice
run_stats=[]
shard_dir=RUN/'shards';shard_dir.mkdir(exist_ok=True)
# Each source writes an isolated gzip shard, renamed atomically on successful completion.
for sid,src in sorted(registry.items()):
    if src.get('kind')=='hf' and not ENABLE_HF_STREAMING: continue
    if src.get('kind') not in ('hf','local'): continue
    fingerprint=sha256(json.dumps(src,sort_keys=True,ensure_ascii=False))
    done=conn.execute('SELECT fingerprint,stats_json FROM completed WHERE source_id=?',(sid,)).fetchone()
    if done:
        if done[0]!=fingerprint: raise RuntimeError(f'Source changed; use new RUN_ID: {sid}')
        run_stats.append(json.loads(done[1]));continue
    # Savepoint allows a failed source to roll back hash inserts.
    conn.execute('SAVEPOINT source_work')
    safe=sha256(sid)[:20]
    tmp=shard_dir/f'{safe}.jsonl.gz.tmp'
    final=shard_dir/f'{safe}.jsonl.gz'
    count=unique=duplicates=flags_count=potential=exported_tokens=0
    group_ids=set()
    try:
        with gzip.open(tmp,'wt',encoding='utf-8') as out:
            for obj in islice(documents(src),PILOT_MAX_DOCUMENTS_PER_SOURCE):
                count+=1
                text=normalize_text(obj['text'])
                if not text:continue
                token_count=proxy_tokens(text)
                potential+=token_count
                h=sha256(text)
                inserted=conn.execute('INSERT OR IGNORE INTO seen(hash,source_id) VALUES (?,?)',(h,sid)).rowcount
                if not inserted:duplicates+=1;continue
                unique+=1
                flags=quality_flags(text)
                if flags:flags_count+=1
                group_id=str(obj.get('document_group') or obj.get('id') or sid)
                group_ids.add(group_id)
                if sid not in approved_ids or flags:continue
                row={'text':text,'source_id':sid,'family_id':src['family_id'],
                     'language':src['language'],'domain':src['domain'],
                     'document_group':group_id,'document_id':obj.get('id'),
                     'title':obj.get('title'),'source_url':obj.get('url') or src.get('source_url'),
                     'revision_id':obj.get('revision_id'),'sha256':h,
                     'tokens_proxy':token_count}
                out.write(json.dumps(row,ensure_ascii=False)+'\n')
                exported_tokens+=token_count
        stats={'source_id':sid,'language':src['language'],'domain':src['domain'],
               'family_id':src['family_id'],'eligible':sid in approved_ids,
               'documents_seen':count,'unique':unique,'exact_duplicates':duplicates,
               'quality_flagged':flags_count,'potential_tokens_proxy':potential,
               'exported_tokens_proxy':exported_tokens,'document_groups':len(group_ids)}
        conn.execute('INSERT INTO completed VALUES (?,?,?)',(sid,fingerprint,json.dumps(stats)))
        conn.execute('RELEASE SAVEPOINT source_work');conn.commit()
        os.replace(tmp,final)
        run_stats.append(stats)
    except Exception:
        conn.execute('ROLLBACK TO SAVEPOINT source_work')
        conn.execute('RELEASE SAVEPOINT source_work');conn.commit()
        if tmp.exists():tmp.unlink()
        raise
    print(sid,':',count,'documents;',potential,'potential tokens;',exported_tokens,'eligible tokens')

stats_df=pd.DataFrame(run_stats)
stats_df.to_csv(RUN/'source_stats.csv',index=False)
print('Total potential:',int(stats_df.potential_tokens_proxy.sum()) if len(stats_df) else 0)
print('Total eligible:',int(stats_df.exported_tokens_proxy.sum()) if len(stats_df) else 0)

**Scaling note:** the default pilot has a per-source cap; `potential_tokens_proxy` is therefore **observed pilot tokens**, not the full 9.86M-token v5 inventory. For billions of tokens, use versioned source partitions and distributed deduplication. SQLite exact hashing is a functional baseline, not a distributed near-duplicate system.

**Important:** an interrupted run between database commit and shard rename can leave an inconsistent shard. The integrity cell below detects this; repair by creating a new `RUN_ID`. Do not rely on a partial run for training.

In [ ]:
# 11 — Integrity checks and source-family split (no leakage across family_id)
completed={r[0]:json.loads(r[1]) for r in conn.execute('SELECT source_id,stats_json FROM completed')}
for sid,stat in completed.items():
    path=shard_dir/(sha256(sid)[:20]+'.jsonl.gz')
    if not path.exists():raise RuntimeError(f'Missing shard for committed source {sid}; create new RUN_ID')
family_to_split={}
for family in sorted({s['family_id'] for s in registry.values() if s['source_id'] in completed and s['source_id'] in approved_ids}):
    # Stable hash, not Python's randomized hash()
    fraction=int(sha256(str(SEED)+'|'+family)[:12],16)/float(16**12)
    family_to_split[family]='test' if fraction<.10 else ('validation' if fraction<.20 else 'train')
print('Approved families:',len(family_to_split),'| split distribution:',dict(Counter(family_to_split.values())))
if len(family_to_split)<3: print('WARNING: insufficient independent approved families for train/val/test')

In [ ]:
# 12 — Merge approved-only shards into split outputs (pilot budget)
split_dir=RUN/'splits';split_dir.mkdir(exist_ok=True)
counts=Counter(); tokens=Counter(); seen_hashes=set()
handles={s:gzip.open(split_dir/f'{s}.jsonl.gz','wt',encoding='utf-8') for s in ('train','validation','test')}
try:
    for sid in sorted(completed):
        if sid not in approved_ids:continue
        src=registry[sid];split=family_to_split[src['family_id']]
        file=shard_dir/(sha256(sid)[:20]+'.jsonl.gz')
        with gzip.open(file,'rt',encoding='utf-8') as f:
            for line in f:
                obj=json.loads(line)
                if obj['sha256'] in seen_hashes:raise RuntimeError('Duplicate export hash')
                seen_hashes.add(obj['sha256'])
                if split=='train' and tokens['train']+obj['tokens_proxy']>PILOT_MAX_TOKENS_TOTAL:continue
                handles[split].write(line)
                counts[split]+=1;tokens[split]+=obj['tokens_proxy']
finally:
    for h in handles.values():h.close()
print('Exported records:',dict(counts),'| proxy tokens:',dict(tokens))

## D. Training-exposure planning

A 21.29B-token training run is a **sampling schedule**. This report compares approved unique-ish pilot tokens to desired exposures and estimates average reuse. High reuse is a warning, not a recommended strategy. Domain/language quotas must be sampled from **eligible train split only**. No synthetic logic examples are mislabeled as independent authentic sources.

In [ ]:
# 13 — Exposure budget and scarcity matrix
quota=[]
train_tokens=Counter()
for sid,stat in completed.items():
    if sid in approved_ids and family_to_split.get(registry[sid]['family_id'])=='train':
        train_tokens[(stat['language'],stat['domain'])]+=stat['exported_tokens_proxy']
for lang,lshare in TARGET_LANG.items():
    for domain,dshare in TARGET_DOMAIN.items():
        target=round(TARGET_TOKENS*lshare*dshare)
        available=train_tokens[(lang,domain)]
        quota.append({'language':lang,'domain':domain,'target_exposure_tokens':target,
                      'approved_train_tokens_proxy':available,
                      'exposure_to_available_ratio':round(target/available,2) if available else None,
                      'missing':available==0})
quota_df=pd.DataFrame(quota)
quota_df.to_csv(RUN/'exposure_budget.csv',index=False)
print(quota_df.to_string(index=False))
print('Total target:',sum(x['target_exposure_tokens'] for x in quota))

## E. Optional tokenizer study — only after approved coverage exists

Use a *balanced* sample of approved **train** text. Compare 32k/40k/50k ByteLevel BPE; report tokens per 1,000 characters by language. This is a diagnostic only, not a production tokenizer. The experiment deliberately refuses to run if PT/EN/ES approved training text is missing.

In [ ]:
# 14 — Optional BPE comparison
if ENABLE_TOKENIZER_EXPERIMENT:
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, normalizers
    lang_text=defaultdict(list)
    with gzip.open(split_dir/'train.jsonl.gz','rt',encoding='utf-8') as f:
        for line in f:
            r=json.loads(line)
            if len(lang_text[r['language']])<3000:lang_text[r['language']].append(r['text'])
    if any(len(lang_text[l])<100 for l in TARGET_LANG):
        raise RuntimeError('Need at least 100 approved train records in each language')
    samples=[s for lang in TARGET_LANG for s in lang_text[lang]]
    results=[]
    for vocab_size in (32000,40000,50000):
        t=Tokenizer(models.BPE(unk_token='[UNK]'))
        t.normalizer=normalizers.NFC()
        t.pre_tokenizer=pre_tokenizers.ByteLevel(add_prefix_space=False)
        t.decoder=decoders.ByteLevel()
        trainer=trainers.BpeTrainer(vocab_size=vocab_size,min_frequency=2,
                 initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
                 special_tokens=['[PAD]','[UNK]','[BOS]','[EOS]'])
        t.train_from_iterator(samples,trainer=trainer)
        t.save(str(RUN/f'bpe_{vocab_size}.json'))
        for lang in TARGET_LANG:
            chars=sum(len(x) for x in lang_text[lang])
            ntokens=sum(len(t.encode(x).ids) for x in lang_text[lang])
            results.append({'vocab_target':vocab_size,'vocab_actual':t.get_vocab_size(),
                 'language':lang,'tokens_per_1000_chars':round(1000*ntokens/max(1,chars),2)})
    pd.DataFrame(results).to_csv(RUN/'tokenizer_comparison.csv',index=False)
    print(pd.DataFrame(results))
else:
    print('Tokenizer experiment OFF. Enable after rights review and balanced data acquisition.')

## F. Audits and manifest

This notebook detects **exact** duplicates only. A production release must additionally pass MinHash/LSH or similar near-duplicate detection, multilingual language identification, contamination screening against frozen benchmarks, quality sampling, and license checks at document level. It does not mark production-ready automatically.

In [ ]:
# 15 — Optional benchmark exact overlap (benchmark must be provided independently)
BENCH=V8/'config/heldout_benchmark.jsonl'
benchmark_count=0;overlap_count=0
if BENCH.exists():
    # Build a set of exact hashes from eligible train split
    train_hashes=set()
    with gzip.open(split_dir/'train.jsonl.gz','rt',encoding='utf-8') as f:
        for line in f:train_hashes.add(json.loads(line)['sha256'])
    with BENCH.open(encoding='utf-8') as f:
        for line in f:
            try: obj=json.loads(line)
            except json.JSONDecodeError:continue
            txt=obj.get('text') or obj.get('prompt')
            if not isinstance(txt,str):continue
            benchmark_count+=1
            if sha256(normalize_text(txt)) in train_hashes:overlap_count+=1
print('Benchmark prompts:',benchmark_count,'| Exact overlaps:',overlap_count)
print('Note: exact hash equality misses embedded, paraphrased, and semantic leakage.')

In [ ]:
# 16 — Manifest with blockers, counts and hashes
import shutil
source_total=len(registry)
eligible_total=len(approved_ids)
observed=sum(x['potential_tokens_proxy'] for x in completed.values())
exported=sum(tokens.values())
missing=[f'{l}/{d}' for l in TARGET_LANG for d in TARGET_DOMAIN
         if train_tokens[(l,d)]==0]
blockers=[]
if eligible_total<source_total:blockers.append('Source and document rights reviews incomplete')
if missing:blockers.append('Approved train coverage missing for: '+', '.join(missing))
if len(family_to_split)<3:blockers.append('Fewer than 3 independent approved source families')
if benchmark_count==0:blockers.append('No frozen benchmark supplied for contamination audit')
if overlap_count:blockers.append('Exact benchmark overlaps found')
if not ENABLE_TOKENIZER_EXPERIMENT:blockers.append('Final tokenizer not trained or certified')
blockers.append('Near-duplicate, multilingual quality, license and semantic leakage audits pending')
manifest={'version':'v8','generated_utc':utc_now(),'run_id':RUN_ID,
 'parameters':354823168,'target_processed_tokens':TARGET_TOKENS,
 'target_language':TARGET_LANG,'target_domain':TARGET_DOMAIN,
 'tokenizer_for_audit':'gpt2 proxy', 'pilot_max_documents_per_source':PILOT_MAX_DOCUMENTS_PER_SOURCE,
 'sources_registered':source_total,'sources_eligible':eligible_total,
 'pilot_observed_potential_tokens_proxy':observed,'pilot_exported_tokens_proxy':exported,
 'split_records':dict(counts),'split_tokens_proxy':dict(tokens),
 'approved_family_count':len(family_to_split),'missing_train_cells':missing,
 'benchmark_prompts':benchmark_count,'benchmark_exact_overlaps':overlap_count,
 'readiness_blockers':blockers,'production_ready':False,'sha256':{}}
for p in sorted(RUN.rglob('*')):
    if p.is_file() and p!=DB and p.suffix!='-wal' and p.name!='manifest_v8.json':
        # Avoid hashing SQLite while open; hash export/report artifacts only.
        manifest['sha256'][str(p.relative_to(RUN))]=hashlib.sha256(p.read_bytes()).hexdigest()
OUT=RUN/'manifest_v8.json'
OUT.write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print('Manifest:',OUT)
print('PRODUCTION READY:',manifest['production_ready'])
for b in blockers:print(' -',b)

## G. Operational roadmap to 21.29B processed tokens

1. **Provenance repair:** recover original page IDs/revision IDs from original snapshot metadata; do not claim a current revision matches archived text without verification.
2. **Acquisition:** add specific licensed/rights-reviewed collections to the registry. Prefer version-pinned bulk datasets and dumps; avoid scraping API pages at scale. Keep source families independent.
3. **Quality:** run language ID, PII/sensitive-data policy review, contamination checks, near-dedup and stratified human quality review.
4. **Tokenizer:** train a final multilingual tokenizer from representative **eligible train** text, freeze vocabulary/special tokens, and recount tokens.
5. **Scaling:** use durable, sharded processing (HPC/SSD), checkpointed ingestion and dataset version manifests. A Colab pilot is not a billion-token production data pipeline.
6. **Training:** record both *unique tokens* and *processed tokens*; enforce per-domain/language sampling, record repeat counts, evaluate at ~5.32B / 10.64B / 15.97B / 21.29B processed tokens.

**Research caution:** 3× Chinchilla is an experiment design choice, not a guarantee of useful conversation. The 25% logic allocation and other domain shares are hypotheses to validate on frozen multilingual benchmarks.